<h2 style="text-align:center;">📘 Chunking in Large Language Models (LLMs)</h2>

---

### 🔹 What is Chunking in LLMs?
In the context of **Large Language Models (LLMs)** such as GPT-2, GPT-3, etc., **chunking** refers to breaking down long pieces of text into smaller, manageable segments (chunks) that fit within the model’s **maximum token limit**.  

👉 Why?  
- LLMs can only process a limited number of tokens at once (e.g., GPT-2 ≈ 1024 tokens).  
- Very long documents exceed this limit and cause errors.  
- Chunking allows us to **split the document**, process each part, and then combine results.

---

### 🔹 Why Do We Need Chunking?
1. **Token Limitations**: Each model has a max context window (e.g., GPT-2 ~1024, GPT-3 ~4096, GPT-4 ~32k).  
2. **Efficiency**: Smaller chunks reduce memory usage and speed up processing.  
3. **Context Preservation**: Focuses the model on a smaller piece of text, reducing confusion.  
4. **Applications**:  
   - Summarization of long articles  
   - Question answering over large documents  
   - Information retrieval  
   - Chatbots handling long histories  

---

### ⚠️ CPU Limitation Notice
Running transformer models like **GPT-2** on CPU can be **extremely slow** or may fail due to memory limits.  
That’s why **GPU acceleration** is recommended.  

✅ Best practice: Run this notebook in **Google Colab** with GPU enabled.  

---

### 🔹 How to Enable GPU in Colab
1. Go to the menu:  
   `Runtime > Change Runtime Type > Hardware Accelerator > GPU`  
2. Then click **Save**.  

You will now have GPU support for faster execution 🚀  

---



In [7]:
# 📦 Install necessary libraries (only in Colab environment)
! pip install transformers torch --quiet


---

### 🔹 Load Pre-trained GPT-2 Model
We’ll use **Hugging Face Transformers** to load GPT-2 (a small but powerful LLM).


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load GPT-2 tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on: {device}")


Model loaded on: cuda


---

### 🔹 Step 1: Define a Chunking Function
This function will split long text into smaller token-limited pieces that GPT-2 can handle.


In [9]:
def chunk_text(text, max_length=512):
    """Split text into chunks of max_length tokens"""
    tokens = tokenizer.encode(text, return_tensors="pt")[0]
    chunks = []

    for i in range(0, len(tokens), max_length):
        chunk = tokens[i:i + max_length]
        chunks.append(chunk)

    return chunks


---

### 🔹 Step 2: Generate Responses for Each Chunk
We now pass each chunk through GPT-2 and generate text outputs.


In [10]:
def generate_responses(chunks, max_new_tokens=50):
    """Generate responses for each chunk using GPT-2"""
    responses = []
    for chunk in chunks:
        input_ids = chunk.unsqueeze(0).to(device)

        # Generate continuation with attention mask + pad_token_id
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,   # instead of max_length
            do_sample=True,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,  # set pad token
            attention_mask=torch.ones_like(input_ids)  # avoid warning
        )

        response = tokenizer.decode(output[0], skip_special_tokens=True)
        responses.append(response)
    return responses


---

### 🔹 Step 3: Example – Chunking a Long Document
We’ll create a **dummy long text** (simulating an article) and process it in chunks.


In [11]:
# Example long text (repeated to simulate large input)
long_text = "Artificial Intelligence is transforming industries. " * 50

# Split into chunks
chunks = chunk_text(long_text, max_length=100)
print(f"Number of chunks created: {len(chunks)}")


Number of chunks created: 4


---

### 🔹 Step 4: Generate Text for Each Chunk
Now we pass each chunk to GPT-2 and collect the generated responses.


In [12]:
responses = generate_responses(chunks, max_new_tokens=50)

# Print responses
for i, response in enumerate(responses, 1):
    print(f"\n📌 Response for Chunk {i}:\n")
    print(response[:500])  # Show first 500 characters




📌 Response for Chunk 1:

Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is trans

📌 Response for Chunk 2:

 transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artificial Intelligence is transforming industries. Artif

---

<h3 style="text-align:center;">✅ Summary</h3>

- **Chunking in NLP vs LLMs**  
  - In NLP: Breaking sentences into **phrases** (NP, VP, PP).  
  - In LLMs: Breaking **long documents into smaller token-limited chunks**.  

- **Why Chunking is Important in LLMs**  
  - Handles documents longer than model’s token limit  
  - Reduces memory usage and speeds up computation  
  - Useful for summarization, Q&A, chatbots, and retrieval  

- **Best Practice**  
  - Always run **transformer-based LLMs on GPU** for efficiency (Colab, Kaggle, or local GPU machines).  

🚀 With chunking, LLMs can now handle **very large documents** piece by piece while still maintaining context!
